# Notebook 11: Anomaly Detection

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 60 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain three major anomaly detection paradigms: **density-based (LOF)**, **isolation-based (Isolation Forest)**, and **boundary-based (One-Class SVM)**
2. Implement Isolation Forest logic from scratch (conceptually) to understand *why* it works
3. Apply all three sklearn implementations to synthetic data and a retail fraud scenario
4. Threshold and evaluate anomaly scores using precision, recall, F1, and ROC curves
5. Select the right method for a given business problem

---

**Week 11 theme:** Customer segmentation / retail data — detecting fraudulent transactions and VIP outliers

## The Airport Security Analogy

Imagine you work at airport security. Over years of watching thousands of passengers, you develop a mental model of what **normal** looks like: check-in → security → gate → boarding. Most passengers follow this predictable path.

Now suppose someone behaves very differently. How would *you* catch them?

- **Isolation Forest** is like a perceptive security guard who can identify a suspicious passenger very quickly — because that passenger takes *unusual paths* through the airport that are easy to isolate. Normal passengers are surrounded by other passengers doing the same thing; isolating one of them requires many careful distinctions. A suspicious passenger is alone in an unusual corner — just one or two observations separates them from everyone else.

- **Local Outlier Factor (LOF)** is like noticing that someone doesn't fit *with their immediate crowd*. Even if a whole terminal is quiet at 3 AM (low overall density), you'd flag someone who seems to have far fewer people around them than even their small cluster of fellow travellers.

- **One-Class SVM** is like drawing a tight boundary fence around every place normal passengers are allowed to be. Anyone found *outside the fence* is flagged — regardless of how far outside or how isolated they are.

All three approaches find anomalies, but they ask fundamentally different questions.

## Why Does Anomaly Detection Matter for ML?

Anomaly detection is one of the most practically valuable skills in the ML toolkit:

| Domain | What counts as an anomaly | Stakes |
|---|---|---|
| **Fraud detection** | Unusual transaction pattern | Direct financial loss |
| **Network intrusion** | Abnormal traffic patterns | Data breach |
| **Manufacturing** | Defective product measurements | Recall costs |
| **Healthcare** | Abnormal patient vitals | Patient safety |
| **Data quality** | Corrupted or mis-entered records | Model degradation |
| **Retail** | Returns fraud, account takeover | Revenue loss |

**The unsupervised challenge:** In most real problems, we do NOT have labelled anomalies. We have lots of normal data and must learn what "normal" looks like — then flag deviations. This is fundamentally different from binary classification.

In [ ]:
# ── Core numerical and data libraries ──────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib_venn import venn3          # pip install matplotlib-venn

# ── Sklearn anomaly detectors ───────────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope

# ── Sklearn utilities ───────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_curve, auc)
from sklearn.datasets import make_blobs

# ── Global plot style ────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid')
np.random.seed(42)

print('All imports successful.')

## Three Paradigms at a Glance

Before writing a single line of code, let's understand the *philosophical difference* between these three families:

### 1. Density-Based: Local Outlier Factor (LOF)
> *Outliers live in sparse, low-density regions compared to their neighbours.*

LOF computes the **local density** around each point by looking at how far away its k nearest neighbours are. It then compares that density to the density around *those neighbours*. If a point is in a much lower-density neighbourhood than its neighbours, it's flagged.

- **Strength:** Handles datasets with varying densities — what's "normal" can differ across the space
- **Weakness:** Slow for large n (O(n²) naive), doesn't produce easy-to-threshold scores

### 2. Isolation-Based: Isolation Forest
> *Outliers are easy to isolate — they need fewer random cuts to separate from all other points.*

Isolation Forest builds many random trees by picking a random feature and a random split value. It counts how many splits are needed to isolate each point. Anomalies need fewer splits (they're alone in unusual corners); normal points need many splits (they're surrounded by similar points).

- **Strength:** Very fast (O(n log n)), handles high-dimensional data, no density estimation
- **Weakness:** Can miss anomalies that cluster together ("micro-clusters" of anomalies)

### 3. Boundary-Based: One-Class SVM
> *Learn a tight boundary (hyperplane / hypersphere in kernel space) that encloses normal data.*

One-Class SVM maps data to a high-dimensional kernel space and finds the tightest boundary that contains most of the training data. Points outside this boundary are anomalies. The **ν** (nu) parameter controls the expected fraction of outliers.

- **Strength:** Works well when you have a clean training set of normal data only; expressive with kernels
- **Weakness:** Sensitive to hyperparameters; slow for large datasets; needs careful normalisation

In [ ]:
np.random.seed(42)

# ── Generate synthetic 2D anomaly detection dataset ─────────────────────────
# 95% of data: a normal Gaussian cluster centred at (0, 0)
n_normal = 950
X_normal = np.random.randn(n_normal, 2)       # standard normal in 2D

# 5% of data: anomalies scattered uniformly in a wider region [-6, 6]²
n_anomaly = 50
X_anomaly = np.random.uniform(-6, 6, size=(n_anomaly, 2))

# Combine into one dataset (labels: 1 = normal, -1 = anomaly)
X_all = np.vstack([X_normal, X_anomaly])
y_true = np.concatenate([np.ones(n_normal), -np.ones(n_anomaly)])  # sklearn convention

# ── Visualise WITHOUT revealing true labels (as in a real problem) ───────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X_all[:, 0], X_all[:, 1],
           alpha=0.5, s=20, color='steelblue', edgecolors='none')
ax.set_title('Synthetic Dataset — Can you spot the anomalies?', fontsize=13)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

print(f'Dataset shape: {X_all.shape}')
print(f'Normal points: {n_normal}  |  Anomalies: {n_anomaly}  |  '
      f'Contamination: {n_anomaly / len(X_all):.1%}')

## Isolation Forest — The Core Idea

Here is the intuition behind Isolation Forest, step by step:

**Step 1 — Build a random tree:**  
Pick a random feature. Pick a random split value between the min and max of that feature. Split the data into two groups. Repeat recursively on each group until each point is isolated (alone in a leaf).

**Step 2 — Measure path length:**  
Count how many splits were needed to isolate each point. Call this the *path length*.

**Step 3 — Anomaly score:**  
Average the path length across many trees. Short path length = easy to isolate = likely anomaly. Long path length = hard to isolate = likely normal.

**Why does this work?**  
Anomalies are in sparse regions of the feature space. A random axis-aligned split is much more likely to isolate them early. Normal points are surrounded by similar points — you need many splits to separate one from its neighbours.

```
Normal point:    ████████████████░░░░ (many splits needed — hard to isolate)
Anomaly:         █░░░░░░░░░░░░░░░░░░░ (very few splits needed — easy to isolate)
```

The anomaly score is normalised so that scores close to **1** are very anomalous and scores close to **0** are very normal. sklearn's `decision_function` returns values where **negative = anomaly**.

In [ ]:
np.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
#  Isolation Forest — From-Scratch Conceptual Implementation
# ════════════════════════════════════════════════════════════════════════════

class IsolationTreeScratch:
    """A single isolation tree (random binary tree that isolates points)."""

    def __init__(self, max_depth=10):
        self.max_depth = max_depth
        self.split_feature = None   # which feature to split on
        self.split_value = None     # the threshold value for the split
        self.left = None            # sub-tree for points <= split_value
        self.right = None           # sub-tree for points > split_value
        self.size = 0               # number of points that reached this node

    def fit(self, X, depth=0):
        self.size = len(X)
        # Stop: too deep OR only one point left (fully isolated)
        if depth >= self.max_depth or len(X) <= 1:
            return self
        # Randomly pick a feature dimension
        self.split_feature = np.random.randint(X.shape[1])
        col = X[:, self.split_feature]
        # Randomly pick a split threshold between min and max of that feature
        self.split_value = np.random.uniform(col.min(), col.max())
        left_mask = col <= self.split_value
        # Recursively build left and right subtrees
        self.left  = IsolationTreeScratch(self.max_depth).fit(X[left_mask],  depth + 1)
        self.right = IsolationTreeScratch(self.max_depth).fit(X[~left_mask], depth + 1)
        return self

    def path_length(self, x, depth=0):
        """Return the depth at which point x gets isolated."""
        # Base case: leaf node (can't split further)
        if self.split_feature is None or self.size <= 1:
            return depth
        # Recurse left or right based on the split
        if x[self.split_feature] <= self.split_value:
            return self.left.path_length(x, depth + 1)
        else:
            return self.right.path_length(x, depth + 1)


class IsolationForestScratch:
    """An ensemble of IsolationTreeScratch models."""

    def __init__(self, n_estimators=100, max_samples=256, random_state=42):
        self.n_estimators = n_estimators
        self.max_samples  = max_samples
        np.random.seed(random_state)

    def fit(self, X):
        self.trees = []
        for _ in range(self.n_estimators):
            # Subsample (anomalies don't need the full dataset to be isolated)
            idx  = np.random.choice(len(X), min(self.max_samples, len(X)), replace=False)
            tree = IsolationTreeScratch(
                max_depth=int(np.ceil(np.log2(self.max_samples)))  # theoretically optimal depth
            )
            tree.fit(X[idx])
            self.trees.append(tree)
        return self

    def score_samples(self, X):
        """Return anomaly scores: higher (less negative) = more anomalous."""
        # Collect path length from each tree for each sample
        path_lengths = np.array(
            [[t.path_length(x) for t in self.trees] for x in X]
        )
        # Negate mean path length: shorter path -> higher anomaly score
        return -path_lengths.mean(axis=1)


print('IsolationForestScratch defined. Fitting on 300-point subset (full fit takes ~30s)...')
# Use a small subset for the from-scratch demo (pure Python is slow)
subset_idx = np.random.choice(len(X_all), 300, replace=False)
X_sub = X_all[subset_idx]
y_sub = y_true[subset_idx]

iforest_scratch = IsolationForestScratch(n_estimators=50, max_samples=64)
iforest_scratch.fit(X_sub)
scores_scratch = iforest_scratch.score_samples(X_sub)

print(f'Scratch scores range: [{scores_scratch.min():.2f}, {scores_scratch.max():.2f}]')
print('Higher scores correspond to anomalies (shorter isolation path).')

In [ ]:
np.random.seed(42)

# ── Visualise scratch IF scores vs sklearn IF scores on the subset ───────────
# sklearn Isolation Forest on same subset
sk_if_sub = IsolationForest(n_estimators=50, contamination=0.05, random_state=42)
sk_if_sub.fit(X_sub)
scores_sklearn_sub = sk_if_sub.decision_function(X_sub)  # more negative = more anomalous

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter plot coloured by scratch score
sc1 = axes[0].scatter(X_sub[:, 0], X_sub[:, 1],
                      c=scores_scratch, cmap='RdYlGn', s=40, alpha=0.8)
axes[0].set_title('From-Scratch IF Score\n(green=normal, red=anomaly)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
plt.colorbar(sc1, ax=axes[0], label='Anomaly score (higher = more anomalous)')

# Scatter plot coloured by sklearn score
sc2 = axes[1].scatter(X_sub[:, 0], X_sub[:, 1],
                      c=scores_sklearn_sub, cmap='RdYlGn', s=40, alpha=0.8)
axes[1].set_title('sklearn IF Score\n(green=normal, red=anomaly)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
plt.colorbar(sc2, ax=axes[1], label='Decision function (more negative = more anomalous)')

plt.suptitle('From-Scratch vs. sklearn Isolation Forest — Score Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Verify correlation: both methods should rank anomalies similarly
corr = np.corrcoef(scores_scratch, -scores_sklearn_sub)[0, 1]  # flip sklearn sign
print(f'Rank correlation between scratch and sklearn scores: {corr:.3f}')
print('A high positive value confirms both methods agree on which points are anomalous.')

In [ ]:
np.random.seed(42)

# ── Apply sklearn IsolationForest to the full dataset ───────────────────────
iforest = IsolationForest(
    n_estimators=200,      # more trees = more stable scores
    contamination=0.05,    # we expect ~5% anomalies
    random_state=42
)
iforest.fit(X_all)

# Predictions: +1 = normal, -1 = anomaly (sklearn convention)
if_labels   = iforest.predict(X_all)
if_scores   = iforest.decision_function(X_all)   # continuous score

# ── Visualise detected anomalies ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

# Plot normal points
mask_normal  = if_labels == 1
mask_anomaly = if_labels == -1
ax.scatter(X_all[mask_normal,  0], X_all[mask_normal,  1],
           color='steelblue', alpha=0.4, s=20, label='Normal (predicted)')
ax.scatter(X_all[mask_anomaly, 0], X_all[mask_anomaly, 1],
           color='crimson',   alpha=0.9, s=60, marker='X', label='Anomaly (predicted)')

ax.set_title('sklearn Isolation Forest — Detected Anomalies', fontsize=13)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
plt.tight_layout()
plt.show()

# Quick evaluation against true labels
p  = precision_score(y_true, if_labels, pos_label=-1)
r  = recall_score(   y_true, if_labels, pos_label=-1)
f1 = f1_score(       y_true, if_labels, pos_label=-1)
print(f'Isolation Forest  →  Precision: {p:.2f}  Recall: {r:.2f}  F1: {f1:.2f}')
print(f'Flagged as anomaly: {mask_anomaly.sum()} points')

## Local Outlier Factor (LOF) — Density Comparison

LOF asks: *"How isolated is this point compared to its neighbours?"*

Here is the step-by-step intuition:

**1. Find k nearest neighbours** for every point.  
**2. Compute reachability distance** from point i to neighbour j:  
$$\text{reach-dist}_k(i, j) = \max(k\text{-dist}(j),\ d(i, j))$$  
In plain English: the reachability distance is at least the k-th neighbour distance of j (this smooths out statistical fluctuations in dense regions).  

**3. Compute local reachability density (LRD)** for point i:  
$$\text{lrd}_k(i) = \frac{k}{\sum_{j \in N_k(i)} \text{reach-dist}_k(i, j)}$$  
High LRD = point is in a dense neighbourhood. Low LRD = sparse neighbourhood.  

**4. Compute LOF score** as the ratio of neighbours' LRD to point i's LRD:  
$$\text{LOF}_k(i) = \frac{\sum_{j \in N_k(i)} \frac{\text{lrd}_k(j)}{\text{lrd}_k(i)}}{k}$$  

**Interpretation:**  
- LOF ≈ 1 → normal (similar density to neighbours)
- LOF >> 1 → anomaly (much lower density than neighbours)

In [ ]:
np.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
#  LOF — From-Scratch Implementation
# ════════════════════════════════════════════════════════════════════════════

def lof_scratch(X, k=20):
    """
    Compute Local Outlier Factor scores for each point in X.
    Returns array of LOF scores (higher = more anomalous).
    """
    n = len(X)

    # Step 1: find k nearest neighbours for every point (k+1 to exclude self)
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(X)
    distances, indices = nbrs.kneighbors(X)

    # k-distance of each point = distance to its k-th nearest neighbour
    k_distances = distances[:, k]   # column index k is the k-th neighbour

    # Step 2: reachability distance from i to j
    def reach_dist(i, j):
        # max of k-dist(j) and actual Euclidean distance i->j
        return max(k_distances[j], np.linalg.norm(X[i] - X[j]))

    # Step 3: local reachability density for each point
    lrd = np.zeros(n)
    for i in range(n):
        knn = indices[i, 1:]   # skip index 0 (self)
        mean_reach = np.mean([reach_dist(i, j) for j in knn])
        lrd[i] = 1.0 / (mean_reach + 1e-10)   # +epsilon avoids division by zero

    # Step 4: LOF = ratio of neighbours' mean LRD to own LRD
    lof = np.zeros(n)
    for i in range(n):
        knn    = indices[i, 1:]
        lof[i] = np.mean([lrd[j] for j in knn]) / (lrd[i] + 1e-10)

    return lof


# Compute on a manageable subset (LOF is O(n²) naive)
print('Computing LOF from scratch on 400-point subset...')
subset_idx2  = np.random.choice(len(X_all), 400, replace=False)
X_lof_sub    = X_all[subset_idx2]
y_lof_sub    = y_true[subset_idx2]

lof_scores_scratch = lof_scratch(X_lof_sub, k=15)

# Compare with sklearn LOF on same subset
# Note: sklearn LOF uses novelty=False for unsupervised mode
sk_lof_sub    = LocalOutlierFactor(n_neighbors=15, contamination=0.05)
sk_lof_labels = sk_lof_sub.fit_predict(X_lof_sub)          # +1 normal, -1 anomaly
sk_lof_scores = -sk_lof_sub.negative_outlier_factor_       # flip sign: higher = more anomalous

corr_lof = np.corrcoef(lof_scores_scratch, sk_lof_scores)[0, 1]
print(f'Rank correlation (scratch vs sklearn LOF): {corr_lof:.3f}')

In [ ]:
np.random.seed(42)

# ── Visualise LOF scores as point size (larger circle = higher LOF) ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clip LOF scores to a sensible range for visualisation
sizes_scratch = np.clip(lof_scores_scratch, 1, 10) * 20
sizes_sklearn  = np.clip(sk_lof_scores, 1, 10) * 20

for ax, sizes, title in zip(
        axes,
        [sizes_scratch, sizes_sklearn],
        ['From-Scratch LOF', 'sklearn LOF']):
    sc = ax.scatter(
        X_lof_sub[:, 0], X_lof_sub[:, 1],
        s=sizes,               # larger = more anomalous
        c=sizes,               # colour mirrors size for visual emphasis
        cmap='YlOrRd',
        alpha=0.6,
        edgecolors='grey', linewidths=0.3
    )
    ax.set_title(f'{title}\n(larger circle = higher LOF score = more anomalous)')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    plt.colorbar(sc, ax=ax, label='LOF score')

plt.suptitle('Local Outlier Factor — Score Visualisation', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Quick sklearn LOF evaluation on subset
p_lof  = precision_score(y_lof_sub, sk_lof_labels, pos_label=-1)
r_lof  = recall_score(   y_lof_sub, sk_lof_labels, pos_label=-1)
f1_lof = f1_score(       y_lof_sub, sk_lof_labels, pos_label=-1)
print(f'LOF (sklearn, subset)  →  Precision: {p_lof:.2f}  Recall: {r_lof:.2f}  F1: {f1_lof:.2f}')

## One-Class SVM — Learning the Boundary of Normality

One-Class SVM (also called SVDD — Support Vector Data Description) works differently from the previous two methods:

**The idea:**  
Instead of comparing points to each other, we learn a *single boundary* that encloses all (or most) of the training data. Anything outside the boundary at test time is flagged as an anomaly.

With an RBF kernel, the boundary is a smooth, non-linear surface in input space — it can wrap tightly around complex normal distributions.

**Key hyperparameters:**

| Parameter | Role | Effect |
|---|---|---|
| `kernel` | Maps to feature space | RBF is most common |
| `nu` | Upper bound on fraction of anomalies / lower bound on support vectors | Higher ν = looser boundary (more anomalies) |
| `gamma` | RBF kernel bandwidth | Higher γ = tighter, more complex boundary |

**When to use One-Class SVM:**
- You have a clean set of normal training data and want to flag anything different
- Your normal data lives in a complex, non-linear manifold
- Dataset is small to medium (it does not scale well to large n)

In [ ]:
np.random.seed(42)

# ── Train on normal data only (as in a realistic anomaly detection scenario) ─
X_train_normal = X_normal   # only normal points for training

# Try three nu values to see how the boundary tightens/loosens
nu_values = [0.05, 0.10, 0.15]
colors    = ['forestgreen', 'darkorange', 'crimson']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Create a grid of points to visualise the decision boundary
xx, yy = np.meshgrid(np.linspace(-7, 7, 300),
                     np.linspace(-7, 7, 300))
grid   = np.c_[xx.ravel(), yy.ravel()]

for ax, nu, col in zip(axes, nu_values, colors):
    oc_svm = OneClassSVM(kernel='rbf', nu=nu, gamma='auto')
    oc_svm.fit(X_train_normal)                           # fit on normal data only
    Z = oc_svm.decision_function(grid).reshape(xx.shape)

    # Shade the "normal" region (decision function >= 0)
    ax.contourf(xx, yy, Z, levels=[Z.min(), 0, Z.max()],
                colors=['#fde8e8', '#e8f5e9'], alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors=col)   # decision boundary

    # Plot normal and anomaly points
    ax.scatter(X_normal[:, 0],  X_normal[:, 1],  s=10, alpha=0.4,
               color='steelblue', label='Normal')
    ax.scatter(X_anomaly[:, 0], X_anomaly[:, 1], s=40, alpha=0.8,
               color='crimson',   label='Anomaly', marker='X')

    preds = oc_svm.predict(X_all)
    n_flagged = (preds == -1).sum()
    ax.set_title(f'One-Class SVM  |  nu={nu}\n{n_flagged} points flagged as anomaly')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_xlim(-7, 7)
    ax.set_ylim(-7, 7)
    if ax == axes[0]:
        ax.legend(fontsize=8)

plt.suptitle('One-Class SVM Decision Boundary for Different nu Values', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Method Comparison

Now that we have seen all three methods in action, let's compare them systematically:

| Property | Isolation Forest | LOF | One-Class SVM |
|---|---|---|---|
| **Core idea** | Short isolation path = anomaly | Low density vs neighbours = anomaly | Outside boundary = anomaly |
| **Speed** | Fast — O(n log n) | Slow — O(n²) naive | Medium (depends on kernel) |
| **High-D data** | Handles well | Suffers from curse of dimensionality | Depends on kernel |
| **Variable density** | Less robust | Excellent — local comparison | Poor — global boundary |
| **Training data** | Needs both normal + some anomalies (contamination) | Same | Normal data only is sufficient |
| **Interpretability** | Moderate (path length) | Low (density ratio) | Very low (kernel space) |
| **Hyperparameters** | contamination, n_estimators | k (neighbours), contamination | nu, gamma, kernel |
| **When to use** | Large datasets, unknown anomaly type | Variable-density clusters | Clean normal set, complex boundary |

**Rule of thumb for retail/fraud detection:**  
Start with **Isolation Forest** (fast, robust, works well in practice). Use **LOF** if your normal data has very different densities in different regions. Use **One-Class SVM** only if you have a clean training set and a complex normal manifold.

In [ ]:
np.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
#  Retail Synthetic Dataset — Simulated Transaction Features
# ════════════════════════════════════════════════════════════════════════════

# Simulate 1,000 retail transactions with two features:
#   - transaction_amount (£)
#   - items_per_basket
n_retail_normal  = 950
n_retail_fraud   = 50

# Normal transactions: moderate spend, 2-10 items
amounts_normal = np.random.lognormal(mean=3.5, sigma=0.5, size=n_retail_normal)  # £~33 avg
items_normal   = np.random.randint(1, 12, size=n_retail_normal).astype(float)

# Fraudulent / anomalous transactions: very high amounts OR suspiciously few items
amounts_fraud  = np.concatenate([
    np.random.uniform(300, 800, 25),   # unusually large transactions
    np.random.uniform(0.01, 1.0, 25)   # micro-transactions (test charges)
])
items_fraud    = np.concatenate([
    np.random.randint(1, 3,  25).astype(float),   # big spend, few items
    np.random.randint(1, 2,  25).astype(float)    # test charges
])

X_retail  = np.column_stack([
    np.concatenate([amounts_normal, amounts_fraud]),
    np.concatenate([items_normal,   items_fraud])
])
y_retail  = np.concatenate([np.ones(n_retail_normal), -np.ones(n_retail_fraud)])

# Scale features for fair comparison
scaler_retail = StandardScaler()
X_retail_sc   = scaler_retail.fit_transform(X_retail)

# ── Apply all three detectors ────────────────────────────────────────────────
# Isolation Forest
if_retail = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
if_retail.fit(X_retail_sc)
if_preds  = if_retail.predict(X_retail_sc)
if_scores_retail = if_retail.decision_function(X_retail_sc)

# LOF
lof_retail = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_preds  = lof_retail.fit_predict(X_retail_sc)

# One-Class SVM (trained on normal-looking transactions)
oc_svm_retail = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
oc_svm_retail.fit(X_retail_sc)
svm_preds     = oc_svm_retail.predict(X_retail_sc)

# ── Create comparison DataFrame ──────────────────────────────────────────────
df_retail = pd.DataFrame({
    'amount':       X_retail[:, 0],
    'items':        X_retail[:, 1],
    'true_label':   y_retail,
    'IF_pred':      if_preds,
    'LOF_pred':     lof_preds,
    'SVM_pred':     svm_preds
})

# Summary: how many each method flags
print('Flagged as anomaly (-1):')
print(f"  Isolation Forest : {(if_preds  == -1).sum()}")
print(f"  LOF              : {(lof_preds == -1).sum()}")
print(f"  One-Class SVM    : {(svm_preds == -1).sum()}")
print(f"  True anomalies   : {n_retail_fraud}")

# Show some flagged records
print('\nSample anomalous transactions flagged by Isolation Forest:')
print(df_retail[df_retail['IF_pred'] == -1][['amount', 'items']].head(10).to_string(index=False))

In [ ]:
np.random.seed(42)

# ── Venn diagram: which transactions does each method flag? ──────────────────
set_if  = set(np.where(if_preds  == -1)[0])
set_lof = set(np.where(lof_preds == -1)[0])
set_svm = set(np.where(svm_preds == -1)[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Venn diagram
try:
    v = venn3([set_if, set_lof, set_svm],
              set_labels=('Isolation Forest', 'LOF', 'One-Class SVM'),
              ax=axes[0])
    axes[0].set_title('Overlap of Flagged Anomalies\n(All 3 Methods)')
except Exception:
    # Fallback if matplotlib-venn not installed
    axes[0].text(0.5, 0.5, 'Install matplotlib-venn\nfor Venn diagram',
                 ha='center', va='center', transform=axes[0].transAxes, fontsize=12)
    axes[0].set_title('Venn diagram (requires matplotlib-venn)')

# Scatter: colour = agreement level
agreement = (-(if_preds == -1).astype(int)
             - (lof_preds == -1).astype(int)
             - (svm_preds == -1).astype(int))   # -3 = all 3 agree it's anomaly
sc = axes[1].scatter(X_retail[:, 0], X_retail[:, 1],
                     c=agreement, cmap='Reds_r', s=20, alpha=0.7)
plt.colorbar(sc, ax=axes[1], label='# methods flagging as anomaly (darker = more)')
axes[1].set_title('Transaction Space — Method Agreement')
axes[1].set_xlabel('Transaction Amount (£)')
axes[1].set_ylabel('Items per Basket')
axes[1].set_xscale('log')   # log scale because amounts span large range

plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
#  Evaluation with Known Ground Truth — ROC Curves
# ════════════════════════════════════════════════════════════════════════════

# For ROC we need continuous scores (not binary predictions)
# LOF: use negative_outlier_factor_ (flip sign so higher = more anomalous)
lof_eval = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_eval.fit_predict(X_retail_sc)
lof_scores_retail = -lof_eval.negative_outlier_factor_

# One-Class SVM: use decision_function (flip sign)
svm_scores_retail = -oc_svm_retail.decision_function(X_retail_sc)

# Convert y_retail: sklearn convention -1=anomaly, +1=normal
# For ROC, we need binary 0/1 labels where 1 = anomaly
y_binary = (y_retail == -1).astype(int)

fig, ax = plt.subplots(figsize=(7, 6))

for scores, name, col in [
        (-if_scores_retail, 'Isolation Forest', 'steelblue'),
        (lof_scores_retail, 'LOF',              'darkorange'),
        (svm_scores_retail, 'One-Class SVM',    'forestgreen')
]:
    fpr, tpr, _ = roc_curve(y_binary, scores)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name}  (AUC = {roc_auc:.3f})', color=col, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier (AUC = 0.50)')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — Anomaly Detection on Retail Data', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

# Precision / Recall / F1 table
print('\nClassification Metrics (contamination = 0.05):')
print(f'{"Method":<20} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('-' * 50)
for preds, name in [
        (if_preds,  'Isolation Forest'),
        (lof_preds, 'LOF'),
        (svm_preds, 'One-Class SVM')
]:
    p  = precision_score(y_retail, preds, pos_label=-1)
    r  = recall_score(   y_retail, preds, pos_label=-1)
    f1 = f1_score(       y_retail, preds, pos_label=-1)
    print(f'{name:<20} {p:>10.3f} {r:>8.3f} {f1:>8.3f}')

## Practical Threshold Selection

In real deployments, you rarely have ground-truth labels. How do you pick the anomaly threshold?

### Option 1 — Business requirement (most common)
"We can investigate at most 2% of all transactions per day." → Set contamination = 0.02.

### Option 2 — Score percentile
Sort all anomaly scores. Flag the bottom 5th percentile (most anomalous 5%) regardless of the absolute score value.

```python
threshold = np.percentile(scores, 5)    # flag bottom 5%
flags = scores < threshold
```

### Option 3 — Elbow in score distribution
Plot the sorted anomaly scores. There is often a natural "elbow" where scores drop sharply — this corresponds to the boundary between normal and anomalous. (Works better for Isolation Forest than LOF.)

### Option 4 — Domain expert validation
Show the top-50 flagged transactions to a fraud analyst. They label them and you tune the threshold to achieve acceptable precision and recall.

**Key insight:** Anomaly detection outputs are almost always the *input to a human review process*, not a final automated decision. Optimise for *precision at low contamination* (every flagged case should be worth investigating).

In [ ]:
np.random.seed(42)

# ── Contamination sensitivity sweep ─────────────────────────────────────────
contaminations = np.arange(0.01, 0.21, 0.01)   # 1% to 20%
n_flagged_list = []
precision_list = []
recall_list    = []

for c in contaminations:
    clf = IsolationForest(n_estimators=200, contamination=c, random_state=42)
    clf.fit(X_retail_sc)
    preds = clf.predict(X_retail_sc)
    n_flagged_list.append((preds == -1).sum())
    precision_list.append(precision_score(y_retail, preds, pos_label=-1, zero_division=0))
    recall_list.append(   recall_score(   y_retail, preds, pos_label=-1, zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Number of flagged transactions
axes[0].plot(contaminations * 100, n_flagged_list, 'o-', color='steelblue')
axes[0].axvline(5, color='crimson', linestyle='--', label='True contamination (5%)')
axes[0].axhline(50, color='grey',   linestyle=':',  label='True # anomalies (50)')
axes[0].set_xlabel('Contamination parameter (%)')
axes[0].set_ylabel('# transactions flagged')
axes[0].set_title('Flagged Count vs. Contamination')
axes[0].legend(fontsize=8)

# Precision
axes[1].plot(contaminations * 100, precision_list, 'o-', color='forestgreen')
axes[1].axvline(5, color='crimson', linestyle='--')
axes[1].set_xlabel('Contamination parameter (%)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision vs. Contamination')
axes[1].set_ylim(0, 1.05)

# Recall
axes[2].plot(contaminations * 100, recall_list, 'o-', color='darkorange')
axes[2].axvline(5, color='crimson', linestyle='--')
axes[2].set_xlabel('Contamination parameter (%)')
axes[2].set_ylabel('Recall')
axes[2].set_title('Recall vs. Contamination')
axes[2].set_ylim(0, 1.05)

plt.suptitle('Isolation Forest — Contamination Sensitivity (Retail Data)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Business scenario annotation
print('Business scenario: "We can investigate 2% of daily transactions."')
idx_2pct = np.argmin(np.abs(contaminations - 0.02))
print(f'  At contamination=0.02: {n_flagged_list[idx_2pct]} transactions flagged')
print(f'  Precision: {precision_list[idx_2pct]:.2f}  |  Recall: {recall_list[idx_2pct]:.2f}')
print(f'  → Every ~{1/precision_list[idx_2pct]:.1f} flagged transaction is truly fraudulent')

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** Isolation Forest builds 100 trees and computes the average path length to isolate a point. Point A has an average path length of 3.2 and Point B has an average path length of 11.7. Which is more likely to be an anomaly, and why?

<details><summary>Show answer</summary>

**Point A is more likely to be an anomaly.** A shorter path length means the point was isolated very early in the tree — it took very few random cuts to separate it from all other points. This happens when a point lives in a sparse, unusual region of the feature space. Point B required many splits to isolate, which means it's surrounded by many similar points — typical of a dense, normal cluster.

</details>

---

**Question 2:** You are applying LOF and Isolation Forest to a dataset where normal data has two very different density clusters (a tight cluster of corporate clients and a sparse cluster of casual customers). Which method would you expect to perform better, and why?

<details><summary>Show answer</summary>

**LOF would likely perform better** in this case. LOF uses *local* density comparison — it compares each point to its immediate neighbours. This means that a point in the sparse casual cluster is only flagged if it is unusually sparse *compared to other casual customers*, not compared to the entire dataset. Isolation Forest uses a global random split strategy, which may incorrectly flag points in the sparse (but normal) casual customer cluster as anomalies, simply because they are far from the dense corporate cluster.

</details>

---

**Question 3:** A fraud team says: "We don't care about missing some fraud — we just need to make sure that every case we flag is actually fraud (so our investigators don't waste time on false alarms)." Should you optimise for high precision or high recall? How would you adjust the contamination parameter?

<details><summary>Show answer</summary>

**Optimise for high precision.** The team wants a low false positive rate — every flag should be a real anomaly. To achieve this, you should use a **low contamination parameter** (e.g., 0.01 or 0.02). This makes the model conservative — it only flags the most extreme anomalies. The trade-off is lower recall (some fraud will be missed), but the flagged cases will be very reliable. Conversely, setting contamination=0.15 or higher would increase recall but flood the team with false alarms, defeating the purpose.

</details>

## Key Takeaways

1. **Three paradigms, three questions:** LOF asks "is this point in a low-density area compared to its neighbours?", Isolation Forest asks "how easy is it to isolate this point with random cuts?", One-Class SVM asks "is this point outside the learned boundary of normality?"

2. **Isolation Forest is usually your first choice** for large, high-dimensional datasets. It's fast, robust, and requires minimal tuning.

3. **LOF handles variable-density data better** than global methods, but it's slow for large n and requires careful choice of k.

4. **One-Class SVM is powerful but fragile** — it requires clean normal training data, careful normalisation, and careful hyperparameter tuning.

5. **The contamination parameter is a business decision**, not a statistical one. Tie it to the operational capacity to investigate flagged cases.

6. **Use multiple methods** when labels are unavailable. High agreement across methods (e.g., all three flag the same transaction) gives much stronger evidence of a true anomaly.

7. **Anomaly detection is rarely the final decision** — it's a triage tool that surfaces cases for human review. Design for high precision at low contamination.

---

**Next up — Notebook 12:** A full end-to-end customer segmentation lab, combining KMeans, hierarchical clustering, DBSCAN, PCA, t-SNE, and GMMs on realistic retail RFM data.